# Week 3 — First Backtest Run

> **PROVISIONAL — mid-price only, fixed-spread cost model.**
>
> This notebook is the Week-3 companion to `reports/week3_first_run.md`.
> It loads the `BacktestResult` produced by the CLI and walks through the
> findings per `docs/RESEARCH_PLAN.md` Week 3:
>
> 1. Did the signal actually fire?
> 2. Does it beat the volume-quartile-matched random control on 48h net PnL?
> 3. Where does the PnL come from — broad edge or a long-tail concentration?
> 4. Recommended Week-3 gate color.
>
> All numbers are **provisional**. Do not circulate. The cost model is
> a fixed-spread placeholder until Workstream E lands the Polygon subgraph.

## 1. Setup & run the pipeline

In [ ]:
from __future__ import annotations

import pandas as pd

from polymarket_political.pipeline import run_backtest
from polymarket_political.report.plots import (
    plot_hit_rate_bar,
    plot_pnl_curve,
    plot_pnl_distribution,
)

# NB: With no `markets_fetcher`/`trades_fetcher`/`prices_fetcher` overrides,
# this will hit the real Polymarket APIs. Expect it to take a while and to
# use the on-disk `data/raw/` cache. Use `--max-markets` via the CLI for a
# fast smoke test.
result = run_backtest(window='training', max_markets=25)
print(f'universe: {result.universe_size} markets')
print(f'events:   {len(result.events)}')
print(f'trades:   {len(result.ledger)} signal, {len(result.control_ledger)} control')
print(f'label:    {result.provisional_label}')

## 2. Signal vs. control — 48h net PnL (provisional)

In [ ]:
from dataclasses import asdict

metrics_df = pd.DataFrame(
    [
        {'group': 'signal', **asdict(result.signal_metrics)},
        {'group': 'control', **asdict(result.control_metrics)},
    ]
)
metrics_df

In [ ]:
# Quick sanity check: does the signal directionally beat control?
beats_mean = result.signal_metrics.mean_pnl > result.control_metrics.mean_pnl
beats_hit = result.signal_metrics.hit_rate >= result.control_metrics.hit_rate
print(f'signal mean_pnl > control mean_pnl: {beats_mean}')
print(f'signal hit_rate  >= control hit_rate: {beats_hit}')

## 3. PnL curve

In [ ]:
signal_pnls = [t.pnl_net_48h for t in result.ledger]
control_pnls = [t.pnl_net_48h for t in result.control_ledger]

plot_pnl_curve(signal_pnls, out_dir='reports')

## 4. Hit rate — signal vs control

In [ ]:
plot_hit_rate_bar(
    {
        'signal': result.signal_metrics.hit_rate,
        'control': result.control_metrics.hit_rate,
    },
    out_dir='reports',
)

## 5. PnL distribution (signal only)

The PRD explicitly calls out long-tail concentration as a risk — one huge
winner can mask an otherwise-unprofitable strategy. Check the histogram and
the `top_decile_pnl_share` in the metrics table above.

In [ ]:
plot_pnl_distribution(signal_pnls, out_dir='reports')

## 6. Top-5 events (by |Δprice|)

In [ ]:
ev_df = result.to_events_frame()
if not ev_df.empty:
    ev_df = ev_df.sort_values('abs_price_move', ascending=False).head(5)
ev_df

## 7. Week 3 Gate

Per `docs/RESEARCH_PLAN.md`:

- 🟢 **Green** — signal beats control on **both** mean PnL and hit rate.
- 🟡 **Amber** — signal beats control on **one** of {mean, hit-rate}. Do a
  parameter sweep in Week 4 and retest.
- 🔴 **Red** — signal loses on both. Thresholds are wrong; Week 4 is a full
  sweep, not iteration.

Remember: **provisional** numbers. The gate color here is indicative until
Workstream E upgrades the cost model.

In [ ]:
from polymarket_political.backtest.__main__ import _gate_recommendation

color, msg = _gate_recommendation(result)
print(f'Gate: {color.upper()} — {msg}')